# Building a Bigram Language Model from Scratch
## Understanding N-grams and Probabilistic Language Models

In this notebook, I will learn how language models work by building one from scratch. I'll focus on **Bigram Language Models**, which predict the next word based on the previous word.

### What I'll Learn:
- What are N-grams (Unigrams and Bigrams)
- How to tokenize text
- How to calculate conditional probabilities
- How to generate sentences using probabilities
- What is the zero-probability problem and how to fix it
- How to smooth probabilities using Laplace smoothing


## Introduction to N-grams

### What is an N-gram?
An **N-gram** is a sequence of N words from a text.

- **Unigram (1-gram)**: A single word
  - Example: "the", "dog", "runs"
  
- **Bigram (2-gram)**: A sequence of two consecutive words
  - Example: "the dog", "dog runs", "runs fast"
  
- **Trigram (3-gram)**: A sequence of three consecutive words
  - Example: "the dog runs", "dog runs fast"

### Why N-grams?
N-grams help us understand word patterns and relationships. In a bigram model:
- We learn what word is likely to follow another word
- We can generate text by sampling from learned probabilities
- We can calculate the probability of a sentence

### Example:
Text: "the dog runs fast the cat runs slowly"

**Unigrams**: [the, dog, runs, fast, the, cat, runs, slowly]

**Bigrams**: [(the, dog), (dog, runs), (runs, fast), (fast, the), (the, cat), (cat, runs), (runs, slowly)]

In [3]:
# Importing required libraries
import re
from collections import defaultdict, Counter
import random
import math

# Define a small training corpus (sample text)
# This is our dataset to learn from
training_corpus = """
machine learning is amazing. AI/Ml is powerful. 
deep learning is a subset of machine learning. 
artificial intelligence includes machine learning and deep learning. 
neural networks are used in deep learning. 
machine learning algorithms learn from data. 
data science uses machine learning. 
deep learning requires lots of data.
I am doing NLP and AI internship at DiyoAI.
"""

print("Training Corpus:")
print(training_corpus)

Training Corpus:

machine learning is amazing. AI/Ml is powerful. 
deep learning is a subset of machine learning. 
artificial intelligence includes machine learning and deep learning. 
neural networks are used in deep learning. 
machine learning algorithms learn from data. 
data science uses machine learning. 
deep learning requires lots of data.
I am doing NLP and AI internship at DiyoAI.



## Text Preprocessing and Tokenization

**Tokenization** is the process of breaking text into individual words (tokens).

We need to:
1. Convert text to lowercase (so "Machine" and "machine" are treated the same)
2. Remove punctuation
3. Split into words
4. Remove empty strings

This creates a **vocabulary** - the set of all unique words in our text.

In [4]:
# Step 1: Convert to lowercase
text_lower = training_corpus.lower()

# Step 2: Remove punctuation using regex
# Regex pattern: keep only letters, spaces, and digits
text_cleaned = re.sub(r'[^a-z0-9\s]', '', text_lower)

# Step 3: Split into tokens (words)
tokens = text_cleaned.split()

# Step 4: Create vocabulary (unique words)
vocabulary = set(tokens)

print(f"Number of tokens (total words): {len(tokens)}")
print(f"Vocabulary size (unique words): {len(vocabulary)}")
print(f"\nVocabulary: {sorted(vocabulary)}")
print(f"\nFirst 20 tokens: {tokens[:20]}")

Number of tokens (total words): 56
Vocabulary size (unique words): 35

Vocabulary: ['a', 'ai', 'aiml', 'algorithms', 'am', 'amazing', 'and', 'are', 'artificial', 'at', 'data', 'deep', 'diyoai', 'doing', 'from', 'i', 'in', 'includes', 'intelligence', 'internship', 'is', 'learn', 'learning', 'lots', 'machine', 'networks', 'neural', 'nlp', 'of', 'powerful', 'requires', 'science', 'subset', 'used', 'uses']

First 20 tokens: ['machine', 'learning', 'is', 'amazing', 'aiml', 'is', 'powerful', 'deep', 'learning', 'is', 'a', 'subset', 'of', 'machine', 'learning', 'artificial', 'intelligence', 'includes', 'machine', 'learning']


## Add START and END Tokens

To model the beginning and end of sentences, we add special tokens:
- **`<START>`**: Marks the beginning of a sentence
- **`<END>`**: Marks the end of a sentence

This allows our model to learn:
- What words are likely to start a sentence
- What words are likely to end a sentence

In [5]:
# Split text into sentences
# We'll assume sentences end with a period (which we removed)
# So we split by looking at the original corpus structure
sentences = training_corpus.strip().split('.')

# Process each sentence
tokenized_sentences = []

for sentence in sentences:
    # Clean the sentence
    sentence_lower = sentence.lower()
    sentence_cleaned = re.sub(r'[^a-z0-9\s]', '', sentence_lower)
    sentence_tokens = sentence_cleaned.split()
    
    # Skip empty sentences
    if sentence_tokens:
        # Add START token at the beginning and END token at the end
        sentence_with_markers = ['<START>'] + sentence_tokens + ['<END>']
        tokenized_sentences.append(sentence_with_markers)

print(f"Number of sentences: {len(tokenized_sentences)}")
print("\nFirst 3 sentences with START and END tokens:")
for i, sent in enumerate(tokenized_sentences[:3]):
    print(f"Sentence {i+1}: {sent}")

# Flatten all tokens for unigram counting
all_tokens = []
for sent in tokenized_sentences:
    all_tokens.extend(sent)

print(f"\nTotal tokens (including START and END): {len(all_tokens)}")

Number of sentences: 9

First 3 sentences with START and END tokens:
Sentence 1: ['<START>', 'machine', 'learning', 'is', 'amazing', '<END>']
Sentence 2: ['<START>', 'aiml', 'is', 'powerful', '<END>']
Sentence 3: ['<START>', 'deep', 'learning', 'is', 'a', 'subset', 'of', 'machine', 'learning', '<END>']

Total tokens (including START and END): 74


##  Building Unigram Counts

A **unigram count** tells us how many times each word appears in the corpus.

We use this to:
- Calculate the probability of a word appearing
- Calculate the denominator for conditional probabilities

In [7]:
# Count unigrams (individual words)
unigram_counts = Counter(all_tokens)

print("Unigram Counts (word frequencies):")
print("\nTop 10 most frequent words:")
for word, count in unigram_counts.most_common(10):
    print(f"  '{word}': {count} times")

print(f"\nTotal unique unigrams: {len(unigram_counts)}")
print(f"Total unigram count (sum): {sum(unigram_counts.values())}")

Unigram Counts (word frequencies):

Top 10 most frequent words:
  '<START>': 9 times
  'learning': 9 times
  '<END>': 9 times
  'machine': 5 times
  'deep': 4 times
  'is': 3 times
  'data': 3 times
  'of': 2 times
  'and': 2 times
  'amazing': 1 times

Total unique unigrams: 37
Total unigram count (sum): 74


## Build Bigram Counts

A **bigram count** tells us how many times each pair of consecutive words appears.

For example, if we have tokens [machine, learning, is, great],
the bigrams are: (machine, learning), (learning, is), (is, great)

We store bigrams in a nested dictionary:
```
bigram_counts = {
    'machine': {'learning': 5, 'is': 2},
    'learning': {'is': 4, 'algorithms': 1},
    ...
}
```

This means: "machine" is followed by "learning" 5 times, and by "is" 2 times.

In [9]:
# Build bigram counts
# A bigram is a pair of consecutive words
bigram_counts = defaultdict(Counter)

# Process each sentence to extract bigrams
for sentence in tokenized_sentences:
    # For each pair of consecutive words in the sentence
    for i in range(len(sentence) - 1):
        word1 = sentence[i]  # First word in pair
        word2 = sentence[i + 1]  # Second word in pair (follows word1)
        
        # Increment the count for this bigram
        bigram_counts[word1][word2] += 1

print("Bigram Counts (consecutive word pairs):")
print("\nFor word '<START>' (words that start sentences):")
for word, count in bigram_counts['<START>'].most_common(10):
    print(f"  '<START>' → '{word}': {count} times")

print("\nFor word 'machine' (words that follow 'machine'):")
for word, count in bigram_counts['machine'].most_common(10):
    print(f"  'machine' → '{word}': {count} times")

print(f"\nTotal unique first words in bigrams: {len(bigram_counts)}")

Bigram Counts (consecutive word pairs):

For word '<START>' (words that start sentences):
  '<START>' → 'machine': 2 times
  '<START>' → 'deep': 2 times
  '<START>' → 'aiml': 1 times
  '<START>' → 'artificial': 1 times
  '<START>' → 'neural': 1 times
  '<START>' → 'data': 1 times
  '<START>' → 'i': 1 times

For word 'machine' (words that follow 'machine'):
  'machine' → 'learning': 5 times

Total unique first words in bigrams: 36


## Calculate Bigram Probabilities

**Bigram probability** is calculated using conditional probability:

$$P(word_2 | word_1) = \frac{Count(word_1, word_2)}{Count(word_1)}$$

This reads as: "The probability of word_2 given word_1 equals the number of times they appear together, divided by the number of times word_1 appears."

### Example:
If "machine" appears 10 times, and is followed by "learning" 5 times:

$$P(learning | machine) = \frac{5}{10} = 0.5$$

This means there's a 50% chance that "learning" follows "machine".

In [11]:
# Build bigram probabilities
# This is a conditional probability table
bigram_probabilities = defaultdict(dict)

# For each word that appears in the corpus
for word1 in bigram_counts:
    # Get the total count of this word
    word1_count = unigram_counts[word1]
    
    # For each word that follows word1
    for word2, count in bigram_counts[word1].items():
        # Calculate probability = count of bigram / count of first word
        probability = count / word1_count
        bigram_probabilities[word1][word2] = probability

print("Bigram Probabilities (P(word2 | word1)):")
print("\nFor '<START>' (probability of sentence starting words):")
for word, prob in sorted(bigram_probabilities['<START>'].items(), key=lambda x: x[1], reverse=True):
    print(f"  P('{word}' | '<START>') = {prob:.4f}")

print("\nFor 'machine' (probability of words following 'machine'):")
for word, prob in sorted(bigram_probabilities['machine'].items(), key=lambda x: x[1], reverse=True):
    print(f"  P('{word}' | 'machine') = {prob:.4f}")

print("\nFor 'learning' (probability of words following 'learning'):")
for word, prob in sorted(bigram_probabilities['learning'].items(), key=lambda x: x[1], reverse=True):
    print(f"  P('{word}' | 'learning') = {prob:.4f}")


Bigram Probabilities (P(word2 | word1)):

For '<START>' (probability of sentence starting words):
  P('machine' | '<START>') = 0.2222
  P('deep' | '<START>') = 0.2222
  P('aiml' | '<START>') = 0.1111
  P('artificial' | '<START>') = 0.1111
  P('neural' | '<START>') = 0.1111
  P('data' | '<START>') = 0.1111
  P('i' | '<START>') = 0.1111

For 'machine' (probability of words following 'machine'):
  P('learning' | 'machine') = 1.0000

For 'learning' (probability of words following 'learning'):
  P('<END>' | 'learning') = 0.4444
  P('is' | 'learning') = 0.2222
  P('and' | 'learning') = 0.1111
  P('algorithms' | 'learning') = 0.1111
  P('requires' | 'learning') = 0.1111


##  Display Most Common Bigrams

Let's see which bigrams (word pairs) appear most frequently in our corpus.

In [12]:
# Find all bigrams and their counts
all_bigrams = []

for word1, word2_counts in bigram_counts.items():
    for word2, count in word2_counts.items():
        all_bigrams.append(((word1, word2), count))

# Sort by frequency
all_bigrams.sort(key=lambda x: x[1], reverse=True)

print("Top 15 Most Common Bigrams:")
print(f"\n{'Rank':<6}{'Bigram':<40}{'Count':<8}{'Probability'}")
print("-" * 70)

for rank, ((word1, word2), count) in enumerate(all_bigrams[:15], 1):
    probability = bigram_probabilities[word1][word2]
    print(f"{rank:<6}('{word1}', '{word2}')<{len(word1)+len(word2)+4}>{count:<8}{probability:.4f}")

Top 15 Most Common Bigrams:

Rank  Bigram                                  Count   Probability
----------------------------------------------------------------------
1     ('machine', 'learning')<19>5       1.0000
2     ('learning', '<END>')<17>4       0.4444
3     ('deep', 'learning')<16>4       1.0000
4     ('<START>', 'machine')<18>2       0.2222
5     ('<START>', 'deep')<15>2       0.2222
6     ('learning', 'is')<14>2       0.2222
7     ('data', '<END>')<13>2       0.6667
8     ('<START>', 'aiml')<15>1       0.1111
9     ('<START>', 'artificial')<21>1       0.1111
10    ('<START>', 'neural')<17>1       0.1111
11    ('<START>', 'data')<15>1       0.1111
12    ('<START>', 'i')<12>1       0.1111
13    ('learning', 'and')<15>1       0.1111
14    ('learning', 'algorithms')<22>1       0.1111
15    ('learning', 'requires')<20>1       0.1111


## Predicting the Next Word

Given a word, we can predict what word is likely to follow using our learned probabilities.

The function below:
1. Looks up all words that follow the given word
2. Returns them sorted by probability (most likely first)
3. Returns alternatives if the word wasn't seen in training

In [13]:
def predict_next_word(current_word, bigram_probs, top_k=5):
    """
    Predict the next word(s) after a given word.
    
    Args:
        current_word (str): The word we're predicting after
        bigram_probs (dict): The bigram probability table
        top_k (int): Number of top predictions to return
    
    Returns:
        list: Top k predicted words with their probabilities
    """
    
    # Check if the word exists in our probabilities
    if current_word not in bigram_probs:
        return []  # No predictions for unseen words
    
    # Get all words that follow this word, sorted by probability
    predictions = sorted(
        bigram_probs[current_word].items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    # Return only the top k predictions
    return predictions[:top_k]

# Test the prediction function
test_words = ['machine', 'learning', 'deep', '<START>']

print("Predicting Next Word:")
print()

for word in test_words:
    predictions = predict_next_word(word, bigram_probabilities, top_k=5)
    
    print(f"After '{word}', the likely next words are:")
    
    if predictions:
        for rank, (next_word, prob) in enumerate(predictions, 1):
            print(f"  {rank}. '{next_word}' (probability: {prob:.4f})")
    else:
        print(f"  No predictions found (word not in training data)")
    
    print()


Predicting Next Word:

After 'machine', the likely next words are:
  1. 'learning' (probability: 1.0000)

After 'learning', the likely next words are:
  1. '<END>' (probability: 0.4444)
  2. 'is' (probability: 0.2222)
  3. 'and' (probability: 0.1111)
  4. 'algorithms' (probability: 0.1111)
  5. 'requires' (probability: 0.1111)

After 'deep', the likely next words are:
  1. 'learning' (probability: 1.0000)

After '<START>', the likely next words are:
  1. 'machine' (probability: 0.2222)
  2. 'deep' (probability: 0.2222)
  3. 'aiml' (probability: 0.1111)
  4. 'artificial' (probability: 0.1111)
  5. 'neural' (probability: 0.1111)



## Generating Sentences

Now we can generate new sentences using our learned probabilities!

**How it works:**
1. Start with the `<START>` token
2. Randomly sample the next word based on learned probabilities
3. Repeat: use the sampled word to predict the next word
4. Stop when we reach `<END>` token

Because we're sampling randomly from probabilities, different runs will generate different sentences!

In [14]:
def generate_sentence(bigram_probs, max_length=20):
    """
    Generate a sentence using bigram probabilities.
    
    Args:
        bigram_probs (dict): The bigram probability table
        max_length (int): Maximum length of generated sentence
    
    Returns:
        str: Generated sentence
    """
    
    # Start with the START token
    current_word = '<START>'
    generated_words = []
    
    # Generate words until we hit END or reach max length
    for _ in range(max_length):
        # Get the possible next words and their probabilities
        if current_word not in bigram_probs:
            break  # Unknown word, stop generation
        
        # Get word-probability pairs
        next_words = list(bigram_probs[current_word].keys())
        probabilities = list(bigram_probs[current_word].values())
        
        # Randomly sample a word based on probabilities
        # random.choices picks weighted random samples
        sampled_word = random.choices(next_words, weights=probabilities, k=1)[0]
        
        # Stop if we hit the END token
        if sampled_word == '<END>':
            break
        
        # Add the word to our sentence
        generated_words.append(sampled_word)
        
        # Move to the next word
        current_word = sampled_word
    
    # Join words into a sentence
    sentence = ' '.join(generated_words)
    return sentence

# Generate multiple sentences
print("Generated Sentences using Bigram Model:")
print()

# Set seed for reproducibility (remove this for different sentences each run)
# random.seed(42)

for i in range(5):
    sentence = generate_sentence(bigram_probabilities, max_length=15)
    print(f"{i+1}. {sentence}")


Generated Sentences using Bigram Model:

1. data
2. deep learning and deep learning
3. aiml is amazing
4. neural networks are used in deep learning
5. machine learning


##  The Zero-Probability Problem

### What is the Zero-Probability Problem?

In our bigram model, what happens if we encounter a word pair that wasn't in our training data?

**Problem:**
- If bigram (word1, word2) never appeared in training, P(word2 | word1) = 0/count = 0
- A probability of 0 means "impossible" - but this is too harsh!
- Many valid word pairs might be rare or missing from limited training data

**Example:**
- Our training data might not have "machine science"
- But "machine science" is a perfectly valid phrase!
- Our model would assign it probability 0, which is wrong

### Why is this a problem?
1. **Limited training data**: We can't see all possible word combinations
2. **Generalization fails**: The model can't handle new combinations
3. **Sentence probability becomes zero**: If any bigram is unseen, the whole sentence probability = 0

### Solution: Smoothing
We'll use **Laplace Smoothing** to give small probabilities to unseen events.

## Laplace (Add-1) Smoothing

**Laplace Smoothing** is a simple technique: add 1 to every count!

### Original Probability Formula:
$$P(word_2 | word_1) = \frac{Count(word_1, word_2)}{Count(word_1)}$$

### Smoothed Probability Formula:
$$P_{smooth}(word_2 | word_1) = \frac{Count(word_1, word_2) + 1}{Count(word_1) + V}$$

Where:
- We add 1 to the numerator (never seen = count 1)
- We add V to the denominator (V = vocabulary size, to normalize)

### What this does:
- Unseen bigrams now have probability 1/Total instead of 0
- Probabilities still sum to 1 (proper probability distribution)
- Common bigrams are less affected
- Rare events get small but non-zero probability

In [15]:
# Get vocabulary size (total unique words)
vocabulary_size = len(unigram_counts)

print(f"Vocabulary size: {vocabulary_size}")
print()

# Build smoothed bigram probabilities
bigram_probabilities_smoothed = defaultdict(dict)

# We need to handle ALL possible word pairs
all_words = list(unigram_counts.keys())

for word1 in all_words:
    word1_count = unigram_counts[word1]
    
    # For every possible word2 in vocabulary
    for word2 in all_words:
        # Get count (0 if never seen together)
        count = bigram_counts[word1].get(word2, 0)
        
        # Apply Laplace smoothing
        smoothed_prob = (count + 1) / (word1_count + vocabulary_size)
        
        bigram_probabilities_smoothed[word1][word2] = smoothed_prob

print("Laplace Smoothing Applied!")
print(f"\nNow we have {len(bigram_probabilities_smoothed)} × {vocabulary_size} possible bigrams")
print(f"Total probability table entries: {len(bigram_probabilities_smoothed) * vocabulary_size}")

Vocabulary size: 37

Laplace Smoothing Applied!

Now we have 37 × 37 possible bigrams
Total probability table entries: 1369


## Compare Probabilities With and Without Smoothing

Let's compare the original and smoothed probabilities to see the effect of Laplace smoothing.

In [20]:
# Test word: 'machine'
test_word = 'machine'

print(f"Comparing probabilities for word '{test_word}':")
print()
print(f"{'Next Word':<15} {'Original P':<20} {'Smoothed P':<20} {'Change'}")
print("-" * 70)

# Get some test words (using words from vocabulary + some unseen words)
test_next_words = ['learning', 'is', 'data', 'algorithms', 'xyz']

for word2 in test_next_words:
    # Original probability
    original_prob = bigram_probabilities[test_word].get(word2, 0.0)
    
    # Smoothed probability - use .get() with default value for unseen words
    if test_word in bigram_probabilities_smoothed:
        smoothed_prob = bigram_probabilities_smoothed[test_word].get(word2, 0.0)
    else:
        smoothed_prob = 0.0
    
    change = smoothed_prob - original_prob
    
    print(f"'{word2}'<{10}  {original_prob:<19.6f}  {smoothed_prob:<19.6f}  {change:+.6f}")

print()
print("Key observations:")
print("- Seen bigrams (learning): probability decreased slightly due to smoothing")
print("- Unseen bigrams (xyz): probability is 0 in original, small positive in smoothed")
print("- This balances the probability distribution")


Comparing probabilities for word 'machine':

Next Word       Original P           Smoothed P           Change
----------------------------------------------------------------------
'learning'<10  1.000000             0.142857             -0.857143
'is'<10  0.000000             0.023810             +0.023810
'data'<10  0.000000             0.023810             +0.023810
'algorithms'<10  0.000000             0.023810             +0.023810
'xyz'<10  0.000000             0.000000             +0.000000

Key observations:
- Seen bigrams (learning): probability decreased slightly due to smoothing
- Unseen bigrams (xyz): probability is 0 in original, small positive in smoothed
- This balances the probability distribution


##  Calculate Sentence Probability

A key task in NLP is to **score** a sentence - how likely is this sentence according to our model?

We calculate sentence probability by **multiplying** bigram probabilities:

$$P(sentence) = P(word_1 | \text{START}) \times P(word_2 | word_1) \times P(word_3 | word_2) \times ... \times P(\text{END} | word_n)$$

### Why multiplication?
Each bigram probability is conditional: "given we're at word_i, what's the probability of word_{i+1}?"

To get the joint probability of the entire sentence, we multiply.

### Log Probability
In practice, we use **log probability** to avoid numerical underflow (very small numbers):

$$\log P(sentence) = \sum \log P(word_i | word_{i-1})$$

This converts multiplication to addition, which is numerically safer.

In [21]:
def calculate_sentence_probability(sentence, bigram_probs, smoothed=False):
    """
    Calculate the probability of a sentence using bigram model.
    
    Args:
        sentence (str): The sentence to score
        bigram_probs (dict): Bigram probability table
        smoothed (bool): Whether to use smoothed probabilities
    
    Returns:
        tuple: (probability, log_probability)
    """
    
    # Tokenize the sentence
    sentence_lower = sentence.lower()
    sentence_cleaned = re.sub(r'[^a-z0-9\s]', '', sentence_lower)
    tokens = sentence_cleaned.split()
    
    # Add START and END tokens
    tokens = ['<START>'] + tokens + ['<END>']
    
    # Initialize log probability
    log_probability = 0.0
    probability = 1.0
    
    # Calculate probability for each bigram
    print(f"\nCalculating probability for: {' '.join(tokens[1:-1])}")
    print()
    print(f"{'Bigram':<30} {'Probability':<15} {'Log Prob'}")
    print("-" * 60)
    
    for i in range(len(tokens) - 1):
        word1 = tokens[i]
        word2 = tokens[i + 1]
        
        # Get probability
        if word1 in bigram_probs and word2 in bigram_probs[word1]:
            prob = bigram_probs[word1][word2]
        else:
            prob = 0.0  # Not seen
        
        # Update log probability
        if prob > 0:
            log_prob = math.log(prob)
            log_probability += log_prob
            probability *= prob
        else:
            log_prob = float('-inf')
            log_probability = float('-inf')
            probability = 0.0
        
        print(f"('{word1}', '{word2}')<{15} {prob:<14.6f}  {log_prob:>10.4f}")
    
    return probability, log_probability

# Test sentences
test_sentences = [
    "machine learning is powerful",
    "deep learning requires data",
    "machine xyz abc",  # Contains unseen bigrams
]

print("="*80)
print("SENTENCE PROBABILITY WITH ORIGINAL (UNSMOOTHED) MODEL:")
print("="*80)

results_original = []
for sentence in test_sentences:
    prob, log_prob = calculate_sentence_probability(sentence, bigram_probabilities, smoothed=False)
    results_original.append((sentence, prob, log_prob))
    print(f"\n→ P(sentence) = {prob:.8f}")
    print(f"→ log P(sentence) = {log_prob:.4f}")
    print()

print("="*80)
print("\nNOTE: Sentences with unseen bigrams have probability 0!")
print("This is the zero-probability problem we discussed.")

SENTENCE PROBABILITY WITH ORIGINAL (UNSMOOTHED) MODEL:

Calculating probability for: machine learning is powerful

Bigram                         Probability     Log Prob
------------------------------------------------------------
('<START>', 'machine')<15 0.222222           -1.5041
('machine', 'learning')<15 1.000000            0.0000
('learning', 'is')<15 0.222222           -1.5041
('is', 'powerful')<15 0.333333           -1.0986
('powerful', '<END>')<15 1.000000            0.0000

→ P(sentence) = 0.01646091
→ log P(sentence) = -4.1068


Calculating probability for: deep learning requires data

Bigram                         Probability     Log Prob
------------------------------------------------------------
('<START>', 'deep')<15 0.222222           -1.5041
('deep', 'learning')<15 1.000000            0.0000
('learning', 'requires')<15 0.111111           -2.1972
('requires', 'data')<15 0.000000              -inf
('data', '<END>')<15 0.666667           -0.4055

→ P(sentence) = 0.0000

In [ ]:
print("="*80)
print("SENTENCE PROBABILITY WITH SMOOTHED MODEL:")
print("="*80)

results_smoothed = []
for sentence in test_sentences:
    prob, log_prob = calculate_sentence_probability(sentence, bigram_probabilities_smoothed, smoothed=True)
    results_smoothed.append((sentence, prob, log_prob))
    print(f"\n→ P(sentence) = {prob:.8f}")
    print(f"→ log P(sentence) = {log_prob:.4f}")
    print()

print("="*80)
print("\nIMPORTANT OBSERVATIONS:")
print("1. Smoothed model can assign non-zero probability to unseen bigrams")
print("2. Even 'nonsense' sentences have small positive probability")
print("3. Real sentences should have higher probability than fake ones")
print("4. This is why smoothing is essential in real NLP systems!")

##  Summary and What I've Learned

### Concepts Learned:

1. **N-grams**: Sequences of N words (unigrams, bigrams, trigrams)
   - Useful for modeling word patterns and relationships

2. **Tokenization**: Breaking text into words and cleaning it
   - Important preprocessing step for NLP

3. **START and END tokens**: Special markers for sentence boundaries
   - Allows model to learn about sentence structure

4. **Conditional Probability**: P(A|B) = "probability of A given B"
   - Formula: Count(A,B) / Count(B)
   - This is the foundation of bigram models

5. **Text Generation**: Sampling from learned probabilities
   - Can create new, plausible text
   - Different each time due to random sampling

6. **Zero-Probability Problem**: Unseen events have probability 0
   - Problem: Can't handle new word combinations
   - This happens with limited training data

7. **Laplace (Add-1) Smoothing**: Simple smoothing technique
   - Add 1 to all counts (numerator and denominator)
   - Ensures all events have non-zero probability
   - Formula: P_smooth = (Count + 1) / (Total + V)

8. **Sentence Probability**: Scoring sentences using learned model
   - Multiply bigram probabilities
   - Use log probability to avoid numerical issues

### Key Insights:

- **Language models learn patterns**: They capture statistical relationships between words
- **Probability is fundamental**: All NLP uses probability in some form
- **Smoothing is essential**: Without it, models fail on new data
- **Simple models work surprisingly well**: Bigrams capture word order surprisingly effectively

### Real-World Applications:

- **Spelling correction**: Find likely corrections for misspelled words
- **Machine translation**: Translate from one language to another
- **Speech recognition**: Find most likely word sequence from audio
- **Autocomplete**: Predict next words (like on your phone keyboard)
- **Text generation**: Create summaries, stories, or content

### Final Thoughts:

This bigram model is simple, but it demonstrates the **core ideas** of natural language processing:
- Extract patterns from data
- Represent those patterns mathematically
- Use learned patterns to solve problems

Modern NLP systems use the same principles, just with more sophisticated models (transformers, RNNs, etc.) and much larger datasets.